## Experience Data Scrapper

In [3]:
from selenium import webdriver
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException
from selenium.common.exceptions import ElementClickInterceptedException
from datetime import time
from datetime import datetime 
import time
import random
import csv
import os

url_list = [
    'https://www.linkedin.com/in/shonbutani/',
    'https://www.linkedin.com/in/katherine-poblete-a6a9895b/',
    'https://www.linkedin.com/in/taro-masuda-78a128176/',
    'https://www.linkedin.com/in/xuanzihe/'
]

# Signing In
driver = webdriver.Chrome()
driver.get("https://www.linkedin.com/login")

username = driver.find_element(By.ID, "username")
password = driver.find_element(By.ID, "password")
button = driver.find_element(By.CLASS_NAME, "btn__primary--large")

username.send_keys("Murtazamajid.123@gmail.com")
password.send_keys("londonbridge")
button.click()


In [4]:
def scroll_down():
    start = time.time()

    initialScroll = 0
    finalScroll = 1000

    while True:
        driver.execute_script(f"window.scrollTo({initialScroll},{finalScroll})")
        initialScroll = finalScroll
        finalScroll += 1000
        
        time.sleep(3)
        
        end = time.time()
        if round(end - start) > 20:
            break

def get_source(profile_url):
    driver.get(profile_url)

    scroll_down()
    
    src = driver.page_source
    soup = BeautifulSoup(src, "html.parser")
    return soup

def scrap_name(soup):
    
    name_div = soup.find('h1', {'class' : 'text-heading-xlarge'})
    name = name_div.text
    
    return name

def scrap_experience():
    #driver.get(profile_url)
    
    experience_list = []
    time.sleep(5)
    tabs = driver.find_elements(By.CLASS_NAME, "artdeco-card")
    for tab in tabs:
        if tab.text:
            tab_text = tab.text
            if isinstance(tab_text, str):
                if tab_text.startswith('Experience'):
                    done = True
                    if tab_text.endswith('experiences'):
                        try:
                            show_all_btn = driver.find_element(By.ID, "navigation-index-see-all-experiences")
                            driver.execute_script("arguments[0].click();", show_all_btn)
                            # show_all_btn.click()
                        except NoSuchElementException:
                            show_all_btn = driver.find_element(By.ID, "navigation-index-see-all-positions-aggregated")
                            driver.execute_script("arguments[0].click();", show_all_btn)
                        time.sleep(3)
                        scroll_down()
                        src = driver.page_source
                        soup = BeautifulSoup(src, "html.parser")
                        experience_elements = soup.find_all('li', {'class' : 'pvs-list__paged-list-item'})
                        for experience_element in experience_elements:
                            try:
                                job_title_select = f'div > div > div.display-flex.flex-column.full-width.align-self-center > div.display-flex.flex-row.justify-space-between > div.display-flex.flex-column.full-width > div > div > div > div > span:nth-child(1)'
                                job_title = experience_element.select_one(job_title_select).text
                                
                                company_name_select = f'div > div > div.display-flex.flex-column.full-width.align-self-center > div.display-flex.flex-row.justify-space-between > div.display-flex.flex-column.full-width > span:nth-child(2)'
                                company_name = experience_element.select_one(company_name_select).text

                                tenure_select = f'div > div > div.display-flex.flex-column.full-width.align-self-center > div.display-flex.flex-row.justify-space-between > div.display-flex.flex-column.full-width > span:nth-child(3) > span.pvs-entity__caption-wrapper'
                                tenure = experience_element.select_one(tenure_select).text

                                experience_instance = job_title + ' | ' + company_name + ' | ' + tenure 
                                experience_list.append(experience_instance)
                            except AttributeError:
                                break
                    else:
                        experience_list.append("Couldn't Scrap")
                        return experience_list
                        scroll_down()
                        src = driver.page_source
                        soup = BeautifulSoup(src, "html.parser")
                        experiences = soup.find_all('li', {'class' : 'artdeco-list__item'})
                        cnt = 1
                        for experience in experiences:
                            try:
                                job_title_select = f'div.pvs-list__outer-container > ul > li:nth-child({cnt}) > div > div.display-flex.flex-column.full-width.align-self-center > div > div.display-flex.flex-column.full-width > div > div > div > div'
                                job_title = experience.select_one(job_title_select).text
        
                                company_name_select = f'div.pvs-list__outer-container > ul > li:nth-child({cnt}) > div > div.display-flex.flex-column.full-width.align-self-center > div > div.display-flex.flex-column.full-width > span:nth-child(2)'
                                company_name = experience.select_one(company_name_select).text
        
                                tenure_select = f'div.pvs-list__outer-container > ul > li:nth-child({cnt}) > div > div.display-flex.flex-column.full-width.align-self-center > div > div.display-flex.flex-column.full-width > span:nth-child(3)'
                                tenure = experience.select_one(tenure_select).text
    
                                experience_instance = job_title + ' | ' + company_name + ' | ' + tenure
                                print(experience_instance)
                                experience_list.append(experience_instance)
                                
                                cnt += 1
                            except AttributeError:
                                break
                    break
                continue
            else:
                continue
        else:
            continue

    return experience_list
    
def add_experience_to_csv(name, profile_url, experience_list):
    field_names = ['Name', 'URL', 'Experience']
    try:
        with open('experience.csv', mode='a', newline='', encoding='utf-8') as file:
            writer = csv.DictWriter(file, fieldnames=field_names)
            
            file.seek(0, 2)
            file_empty = file.tell() == 0
    
            if file_empty:
                writer.writeheader()
    
            for experience in experience_list:
                writer.writerow({'Name' : name, 'URL': profile_url, 'Experience': experience})
            if "Couldn't Scrap" in experience_list:
                print(f"There were no experiences")
            else:
                print(f"Data Added Successfully!")
            
            
    except IOError:
            print("Data could not be added!")
        

def main():
    with open('cleanedUrls.csv', mode='r', encoding='utf-8') as file:
        csv_reader = csv.DictReader(file) 
        for index, row in enumerate(csv_reader):
            if index < 1197:
                continue 
            
            profile_url = row['URLs']  # Assuming 'URLs' is the header for URLs in the CSV file
            soup = get_source(profile_url)
            name = scrap_name(soup)
            
            experience_list = scrap_experience()
            add_experience_to_csv(name, profile_url, experience_list)
        
main()


AttributeError: 'NoneType' object has no attribute 'text'

## Experience Data Cleaning

In [ ]:
import pandas as pd

exp_dataframe = pd.read_csv("experience.csv")
pro_dataframe = pd.read_csv("profile_data.csv")
pro_dataframe = pro_dataframe[["ID", "URL"]]

df = pd.merge(exp_dataframe, pro_dataframe, on=["URL"])

df = df[["ID", "Experience"]]

df1 = df["Experience"].str.split("|", expand=True)

df1[0].str.strip()